# Hello TuiML

**Train, compare, and serve ML models in 5 minutes.**

TuiML gives you three levels of API:
1. **High-Level** — One-liner functions (`tuiml.train()`)
2. **Mid-Level** — Fluent workflow builder (`Workflow().train().run()`)
3. **Low-Level** — Direct class imports for full control

This quickstart covers Level 1 — the fastest path from zero to a trained model.

## 1. Train a Model (One Line)

In [1]:
import tuiml

# Train a Random Forest on the built-in iris dataset with 5-fold cross-validation
result = tuiml.train("RandomForestClassifier", "iris", "class", cv=5)

print(f"Accuracy: {result.metrics['cv_accuracy_score_mean']:.4f}")

Accuracy: 0.9600


That's it. One function call to load data, train, and evaluate.

The `result` object contains everything:

In [2]:
print("Model:", type(result.model).__name__)
print("Metrics:", result.metrics)
print("Metadata:", result.metadata)

Model: RandomForestClassifier
Metrics: {'cv_accuracy_score_mean': 0.9600000000000002, 'cv_accuracy_score_std': 0.024944382578492935, 'cv_f1_score_mean': 0.9395583160800551, 'cv_f1_score_std': 0.04058971815950488}
Metadata: {'algorithm': 'RandomForestClassifier', 'preprocessing': [], 'feature_selection': None, 'evaluation_method': 'cross_validate'}


## 2. Compare 5 Algorithms (One Call)

In [3]:
exp = tuiml.experiment(
    algorithms=[
        "RandomForestClassifier",
        "NaiveBayesClassifier",
        "SVC",
        "KNearestNeighborsClassifier",
        "C45TreeClassifier",
    ],
    datasets=["iris"],
    cv=10,
)

print(exp.summary())

Experiment: Experiment
Validation: cross_validation
Metric: accuracy

Dataset: iris
--------------------------------------------------
  RandomForestClassifier: 0.9467 ± 0.0499
  NaiveBayesClassifier: 0.9467 ± 0.0581
  SVC: 0.9600 ± 0.0442
  KNearestNeighborsClassifier: 0.9533 ± 0.0521
  C45TreeClassifier: 0.9267 ± 0.0629



## 3. Discover What’s Available

In [4]:
classifiers = tuiml.list_algorithms(type="classifier")
regressors = tuiml.list_algorithms(type="regressor")
clusterers = tuiml.list_algorithms(type="clusterer")

print(f"Classifiers: {len(classifiers)}")
print(f"Regressors:  {len(regressors)}")
print(f"Clusterers:  {len(clusterers)}")
print(f"Total:       {len(classifiers) + len(regressors) + len(clusterers)}")

Classifiers: 42
Regressors:  20
Clusterers:  8
Total:       70


In [5]:
# Search for algorithms by keyword
results = tuiml.search_algorithms("forest")
for algo in results:
    print(f"  {algo['name']}")

  DecisionStumpClassifier
  C45TreeClassifier
  RandomTreeClassifier
  RandomForestClassifier
  HoeffdingTreeClassifier
  LogisticModelTreeClassifier
  BaggingClassifier
  RandomSubspaceClassifier
  IsolationForestDetector
  LocalOutlierFactorDetector
  EllipticEnvelopeDetector
  OneClassSVMDetector
  ABODDetector
  BootstrapFeaturesSelector


In [6]:
# Get parameter schema for any algorithm
info = tuiml.describe_algorithm("RandomForestClassifier")
print("Parameters:")
for param, schema in info["parameters"].items():
    print(f"  {param}: {schema.get('type', '?')} = {schema.get('default', '?')}")

Parameters:
  n_estimators: integer = 100
  max_features: ['string', 'integer', 'number'] = sqrt
  max_depth: integer = None
  min_samples_split: integer = 2
  min_samples_leaf: integer = 1
  bootstrap: boolean = True
  oob_score: boolean = False
  random_state: integer = None
  n_jobs: integer = 1


## 4. Save & Load Models

In [7]:
import numpy as np

# Save
tuiml.save(result.model, "my_model.pkl")

# Load and predict
model = tuiml.load("my_model.pkl")
prediction = tuiml.predict(model, np.array([[5.1, 3.5, 1.4, 0.2]]))
print(f"Prediction: {prediction}")

# Clean up
import os; os.remove("my_model.pkl")

Prediction: [0]


## 5. Preprocessing Presets

In [8]:
# Available presets: minimal, fast, standard, full, imbalanced
result = tuiml.train(
    "RandomForestClassifier",
    "diabetes",
    "class",
    preset="standard",  # Impute + Scale + Encode
    cv=5,
)
print(f"Diabetes accuracy: {result.metrics['cv_accuracy_score_mean']:.4f}")

Diabetes accuracy: 0.6719


## 6. Workflow Builder

For more control, chain operations with the fluent API:

In [9]:
from tuiml import Workflow

result = (
    Workflow("iris", target="class")
    .normalize()
    .train("RandomForestClassifier", n_estimators=100)
    .cross_validate(cv=5)
    .run()
)
print(f"Workflow accuracy: {result.metrics['cv_accuracy_score_mean']:.4f}")

Workflow accuracy: 0.9600
